# Simulación de campo de temperatura de rollos en una horno tipo campana durante el procedimiento de recocido. 

Implementa la librería descrita en los archivos de Python.

## Imports

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from Simulator import Simulator
from Furnace import Furnace
from Material import Material
from Coil import Coil
from ProtectionGas import ProtectionGas
from Analyzer import Analyzer

## Creación de objetos

In [2]:
# Steel SPCC-PSR
material = Material(
    density=7870.0,
    emissivity=0.08554,
    specific_heat=481.0,
    hardness=1133.86,
    roughness=3.2215e-6,
    conductivity=73.3
)

gas = ProtectionGas(
   conductivity=0.1805,
   viscosity=8.9e-6,
   density=0.0819
)

coil = Coil(
    material=material,
    inner_radius=0.508 / 2,
    outer_radius=1.761  / 2,
    strip_thickness=1.5e-3,
    height=1.29,
    point_separation=0.05,
    initial_temperature=(273.15 + 55),
    surface_slope=0.1,
    oxide_thickness=10e-6,
    oxide_conductivity=3.0,
    emulsion_correction=0.8
)

# Import sample temperature from file and clean it up
gas_temp = np.genfromtxt("./ExperimentalData/Thermocouples/1#.csv", delimiter=",")
gas_temp = gas_temp[~np.isnan(gas_temp).any(axis=1)]


furnace = Furnace(
    temperature=gas_temp,
    coils=[coil],
    gas=gas,
    gas_velocity=8.0,
    hydraulic_diameter=0.1,
    circulation_length=1.0,
    vortex_enhancement=1.0,
    radiation_enhancement=1.1,
    side_flow_fraction=1.0,
    top_flow_fraction=1.0
)

simulator = Simulator(
    furnace = furnace, 
    time_step = 30.0, 
    simulation_time = furnace.duration
)

## Simulate
Activar simulator.run().

In [3]:
results = simulator.run()

## Analisis

### Settings.

In [22]:
analyzer = Analyzer(results=results,
                    simulator=simulator, 
                    furnace=furnace)

r, z = 1e-3, 0.7
data = analyzer.get_point_data(r, z)
plot_height = 700

layout = {
    "title": {
        "text": f"Evolución de temperatura de punto en (z,r) = {z,r}",
        "x": 0.5,
        "xanchor": "center",
        "y": 0.97,
        "yanchor": "top",
    },
    #"width": 1920,
    "height": plot_height,
    "font": {
        "family": "Arial",
        "size": 15,
        "color": "black",
    },
    "margin": {"t": int(plot_height * 0.15), "b": 100, "l": 100, "r": 100},
    "legend": {"x": 1.05, "xanchor": "left"},
    "xaxis": {"title": {"text": "Tiempo (h)"}, "ticklabelstandoff": 15},
    "yaxis": {"ticklabelstandoff": 15},
    "template": "none"
}

fig = go.Figure(layout=layout)
fig.add_trace(go.Scatter(
    x=simulator.times / 3600.0,
    y=data,
    name="Punto objetivo"
))
fig.add_trace(go.Scatter(
    x=simulator.times / 3600.0,
    y=[furnace.temperature_at(time) for time in simulator.times],
    line = {"width": 2, 
            "dash": 'dash'},
    name="Gas de protección"
));

### Plot.

In [23]:
fig.show()